# Pre-trained DimeNet models on QM9

Evaluate the official pre-trained DimeNet ([Gasteiger et al., 2020](https://arxiv.org/abs/2003.03123))
or DimeNet++ models on QM9: one model per target property. DimeNet passes messages between the bonds
of a molecule and uses both distances and the angles between bonds. `from_qm9_pretrained` downloads
the authors' weights (TensorFlow checkpoints, read with TensorFlow) and returns the data split they used.

Same model as PyG's [`examples/qm9_pretrained_dimenet.py`](https://github.com/pyg-team/pytorch_geometric/blob/master/examples/qm9_pretrained_dimenet.py).

In [ ]:
# Install K3-Node, then choose a backend: "tensorflow", "torch" or "jax"
!pip install -q "k3-node[examples] @ git+https://github.com/anas-rz/k3-node"
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

## Load the data

The DimeNet models order the QM9 targets differently, so the targets are reordered as in PyG's example.

In [ ]:
import numpy as np
from keras import ops
from k3_node.datasets import QM9
from k3_node.loader import DataLoader
from k3_node.models import DimeNet, DimeNetPlusPlus
from k3_node.training import no_grad

use_dimenet_plus_plus = False
Model = DimeNetPlusPlus if use_dimenet_plus_plus else DimeNet

dataset = QM9("data/QM9")
dataset._data.y = ops.convert_to_numpy(dataset._data.y)[:, [0, 1, 2, 3, 4, 5, 6, 12, 13, 14, 15, 11]]

## Evaluate every target

Target 4 (the HOMO-LUMO gap) has no model: it is the difference of targets 3 and 2. Energies are in meV. DimeNet is expensive (it works on every pair of bonds), so to keep the example fast on a CPU only the first 1,000 test molecules are evaluated.

In [ ]:
num_test = 1000  # test molecules per target; None evaluates all of them (best on a GPU)

for target in [t for t in range(12) if t != 4]:
    model, (train_dataset, val_dataset, test_dataset) = Model.from_qm9_pretrained("data/QM9", dataset, target)
    errors = []
    for data in DataLoader(test_dataset[:num_test], batch_size=64):
        with no_grad():  # evaluation only: nothing to remember for gradients
            prediction = model(data.z, data.pos, data.batch, batch_size=data.num_graphs)[:, 0]
        errors.append(np.abs(ops.convert_to_numpy(prediction) - ops.convert_to_numpy(data.y)[:, target]))
    errors = np.concatenate(errors)
    if target in [2, 3, 4, 6, 7, 8, 9, 10]:  # energies: report meV instead of eV
        errors = 1000 * errors
    print(f"Target {target:02d}: MAE {errors.mean():.5f} ± {errors.std():.5f}")